# ChordFlow — Sol El Model Eğitimi

Bu notebook, `landmarks.csv` içindeki 21×3 MediaPipe koordinatından `Unknown (0)` ve 1–7 kök nota derecesini sınıflandıran MLP modelini adım adım eğitir.

Akış:

1. Dataset kontrolü ve sekiz sınıfın dağılımı
2. Bilek merkezleme, avuç ölçekleme ve rotasyon normalizasyonu
3. Temporal blok tabanlı train/validation/test split
4. PyTorch MLP eğitimi ve early stopping
5. Test metrikleri, confusion matrix ve confidence analizi
6. Model ve preprocessing değerlerinin kaydedilmesi

> Notebook'u proje kökünde Jupyter açarak çalıştır. Ardışık kamera karelerinin veri sızıntısını azaltmak için rastgele satır split'i kullanılmaz.

In [1]:
from pathlib import Path
from datetime import datetime
from dataclasses import asdict
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch import nn

# Notebook proje kökünden veya model/ klasöründen açılabilir.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "model" / "data" / "left" / "landmarks.csv").exists():
    if (PROJECT_ROOT / "data" / "left" / "landmarks.csv").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("model/data/left/landmarks.csv bulunamadı.")

MODEL_DIR = PROJECT_ROOT / "model"
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import left_hand_model_training as training

CSV_PATH = MODEL_DIR / "data" / "left" / "landmarks.csv"
OUTPUT_ROOT = MODEL_DIR / "training_outputs" / "left_hand"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Proje: {PROJECT_ROOT}")
print(f"Dataset: {CSV_PATH}")
print(f"Device: {DEVICE}")

Proje: C:\Users\Ulvi\Desktop\ChordFlow
Dataset: C:\Users\Ulvi\Desktop\ChordFlow\model\data\left\landmarks.csv
Device: cpu


## 1. Eğitim ayarları

Aşağıdaki hücre notebook üzerinden değiştireceğin temel hiperparametreleri içerir. `temporal_block_size`, art arda çekilmiş benzer karelerin farklı splitlere dağılmasını azaltır.

In [2]:
config = training.TrainingConfig(
    seed=42,
    epochs=200,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout=0.25,
    patience=25,
    temporal_block_size=20,
    temporal_gap_seconds=2.0,
    validation_ratio=0.15,
    test_ratio=0.15,
    augmentation_noise_std=0.015,
)

training.set_reproducible_seed(config.seed)
config

TrainingConfig(seed=42, epochs=200, batch_size=64, learning_rate=0.001, weight_decay=0.0001, dropout=0.25, patience=25, temporal_block_size=20, temporal_gap_seconds=2.0, validation_ratio=0.15, test_ratio=0.15, augmentation_noise_std=0.015)

## 2. Dataset kontrolü

CSV; sınıf etiketi, fotoğraf yolu, handedness skoru ve 21 landmark için ham `x, y, z` değerlerini içerir. Bu hücre eksik sınıf/kolon ve geçersiz numerik değer kontrolünü de çalıştırır.

In [3]:
frame = training.load_dataset(CSV_PATH)
class_counts = frame["label"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot.bar(ax=axes[0], color="#7c3aed")
axes[0].set(title="Sınıf dağılımı", xlabel="Derece", ylabel="Örnek sayısı")
axes[0].grid(axis="y", alpha=0.25)

frame.boxplot(column="handedness_score", by="label", ax=axes[1])
axes[1].set(title="MediaPipe handedness confidence", xlabel="Derece", ylabel="Confidence")
fig.suptitle("")
fig.tight_layout()
plt.show()

print(f"Toplam örnek: {len(frame)}")
display(class_counts.rename("sample_count").to_frame())

Toplam örnek: 1602


C:\Users\Ulvi\AppData\Local\Temp\ipykernel_21296\698210985.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,sample_count
label,
1,281
2,260
3,261
4,254
5,268
6,278


## 3. Geometrik normalizasyon ve split

Her örnekte:

- Landmark 0 (wrist) koordinat merkezi yapılır.
- El, wrist–middle MCP mesafesiyle ölçeklenir.
- Wrist–middle MCP ekseni yukarı bakacak şekilde XY rotasyonu uygulanır.
- Aykırı koordinatlar `[-4, 4]` aralığında sınırlandırılır.
- Daha sonra yalnızca **train splitinden** hesaplanan mean/std kullanılır.

Train/validation/test ayrımı, aynı kamera burstündeki yakın karelerin veri sızıntısı oluşturmasını azaltmak için temporal bloklarla yapılır.

In [4]:
raw_features = frame[training.landmark_columns()].to_numpy(dtype=np.float32)
features = training.geometric_normalize(raw_features)
labels = frame["label"].map(
    {label: index for index, label in enumerate(training.CLASS_LABELS)}
).to_numpy(dtype=np.int64)

splits = training.split_by_temporal_blocks(frame, config)
split_summary = training.make_split_summary(frame, splits)
loaders, feature_mean, feature_std = training.build_loaders(
    features, labels, splits, config
)

print("Feature shape:", features.shape)
print("Finite values:", np.isfinite(features).all())
display(split_summary.pivot(index="label", columns="split", values="sample_count"))

Feature shape: (1602, 63)
Finite values: True


split,test,train,validation
label,,,
1,32,209,40
2,40,180,40
3,40,200,21
4,27,200,27
5,40,188,40
6,40,198,40


In [5]:
# Ham ve normalize edilmiş aynı örneği karşılaştır.
sample_index = 0
raw_hand = raw_features[sample_index].reshape(21, 3)
normalized_hand = features[sample_index].reshape(21, 3)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for axis, hand, title in zip(
    axes,
    (raw_hand, normalized_hand),
    ("Ham MediaPipe koordinatları", "Geometrik normalize"),
):
    axis.scatter(hand[:, 0], -hand[:, 1], c=np.arange(21), cmap="viridis", s=38)
    for index, (x_value, y_value) in enumerate(hand[:, :2]):
        axis.annotate(str(index), (x_value, -y_value), fontsize=7)
    axis.set_title(title)
    axis.set_aspect("equal", adjustable="datalim")
    axis.grid(alpha=0.2)

fig.tight_layout()
plt.show()

C:\Users\Ulvi\AppData\Local\Temp\ipykernel_21296\3956984152.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Model eğitimi

MLP mimarisi:

```text
63 → Linear(128) → BatchNorm → ReLU → Dropout
   → Linear(64)  → ReLU → Dropout
   → Linear(8 logits)
```

Çıkışlar `Unknown (0)` ve derece 1–7'dir. Loss olarak class-weighted cross entropy, optimizer olarak AdamW kullanılır. Validation loss iyileşmezse early stopping eğitimi durdurur.

In [6]:
model, history, best_epoch = training.train_model(
    loaders,
    labels[splits["train"]],
    config,
    DEVICE,
)

print(f"En iyi epoch: {best_epoch}")
print(f"Tamamlanan epoch: {len(history['train_loss'])}")

Epoch 001 | train loss 1.5290 acc 0.486 | val loss 1.2841 acc 0.707
Epoch 002 | train loss 1.0228 acc 0.683 | val loss 0.8680 acc 0.683
Epoch 003 | train loss 0.6792 acc 0.782 | val loss 0.6667 acc 0.784
Epoch 004 | train loss 0.5049 acc 0.843 | val loss 0.5085 acc 0.894
Epoch 005 | train loss 0.3539 acc 0.898 | val loss 0.3529 acc 0.971
Epoch 006 | train loss 0.2433 acc 0.943 | val loss 0.2716 acc 0.952
Epoch 007 | train loss 0.1946 acc 0.961 | val loss 0.1917 acc 0.981
Epoch 008 | train loss 0.1409 acc 0.976 | val loss 0.1723 acc 0.976
Epoch 009 | train loss 0.1228 acc 0.973 | val loss 0.1480 acc 0.976
Epoch 010 | train loss 0.1005 acc 0.980 | val loss 0.1456 acc 0.966
Epoch 011 | train loss 0.0889 acc 0.979 | val loss 0.1181 acc 0.976
Epoch 012 | train loss 0.0889 acc 0.975 | val loss 0.1124 acc 0.981
Epoch 014 | train loss 0.0610 acc 0.986 | val loss 0.0974 acc 0.976
Epoch 015 | train loss 0.0517 acc 0.991 | val loss 0.0844 acc 0.981
Epoch 018 | train loss 0.0427 acc 0.993 | val lo

In [7]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["validation_loss"], label="Validation")
axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(epochs, history["train_accuracy"], label="Train")
axes[1].plot(epochs, history["validation_accuracy"], label="Validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.02))
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

C:\Users\Ulvi\AppData\Local\Temp\ipykernel_21296\4084308192.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Test değerlendirmesi

Test split eğitim ve model seçimi sırasında kullanılmaz. Accuracy yanında macro F1, sınıf bazlı precision/recall, confusion matrix ve confidence threshold analizi incelenir.

In [8]:
criterion = nn.CrossEntropyLoss()
test_loss, test_accuracy, test_probabilities, test_targets = training.evaluate_loader(
    model,
    loaders["test"],
    criterion,
    DEVICE,
)
test_predictions = test_probabilities.argmax(axis=1)
macro_f1 = f1_score(test_targets, test_predictions, average="macro")

report = classification_report(
    test_targets,
    test_predictions,
    labels=np.arange(len(training.CLASS_LABELS)),
    target_names=training.CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
report_frame = pd.DataFrame(report).transpose()

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")
print(f"Test macro F1: {macro_f1:.4%}")
display(report_frame)

Test loss: 1.0359
Test accuracy: 90.4110%
Test macro F1: 89.9757%


,precision,recall,f1-score,support
degree_1,0.689655,0.62500,0.655738,32.00000
degree_2,0.769231,1.00000,0.869565,40.00000
degree_3,1.000000,1.00000,1.000000,40.00000
degree_4,1.000000,1.00000,1.000000,27.00000
degree_5,1.000000,1.00000,1.000000,40.00000
degree_6,1.000000,0.77500,0.873239,40.00000
accuracy,0.904110,0.90411,0.904110,0.90411
macro avg,0.909814,0.90000,0.899757,219.00000
weighted avg,0.912503,0.90411,0.902721,219.00000


In [9]:
matrix = confusion_matrix(
    test_targets,
    test_predictions,
    labels=np.arange(len(training.CLASS_LABELS)),
)

class_tick_labels = ["Unknown", *training.CLASS_LABELS[1:]]
fig, axis = plt.subplots(figsize=(8, 7))
image = axis.imshow(matrix, cmap="Purples")
fig.colorbar(image, ax=axis)
axis.set(
    xticks=np.arange(len(training.CLASS_LABELS)),
    yticks=np.arange(len(training.CLASS_LABELS)),
    xticklabels=class_tick_labels,
    yticklabels=class_tick_labels,
    xlabel="Tahmin edilen sınıf",
    ylabel="Gerçek sınıf",
    title="Test confusion matrix",
)
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, matrix[row, column], ha="center", va="center")
fig.tight_layout()
plt.show()

C:\Users\Ulvi\AppData\Local\Temp\ipykernel_21296\936725022.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
thresholds = training.threshold_analysis(test_targets, test_probabilities)
confidence = test_probabilities.max(axis=1)
correct = test_predictions == test_targets

fig, axis = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, 1, 21)
axis.hist(confidence[correct], bins=bins, alpha=0.7, label="Doğru", color="#7c3aed")
if (~correct).any():
    axis.hist(confidence[~correct], bins=bins, alpha=0.7, label="Yanlış", color="#e11d48")
axis.set(
    title="Softmax confidence dağılımı",
    xlabel="En yüksek probability",
    ylabel="Örnek sayısı",
)
axis.legend()
axis.grid(alpha=0.2)
plt.show()

display(thresholds)

C:\Users\Ulvi\AppData\Local\Temp\ipykernel_21296\2630258629.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,threshold,coverage,accepted_samples,accepted_accuracy
0,0.50,1.000000,219,0.904110
1,0.60,1.000000,219,0.904110
2,0.70,0.995434,218,0.903670
3,0.80,0.981735,215,0.906977
4,0.85,0.977169,214,0.906542
5,0.90,0.968037,212,0.905660
6,0.95,0.954338,209,0.913876


## 6. Model ve raporları kaydet

Her notebook çalıştırması ayrı timestamp klasörüne yazılır. `.pt` checkpoint preprocessing değerlerini de içerir; `preprocessing.npz` aynı değerleri NumPy formatında ayrıca saklar.

In [11]:
run_directory = OUTPUT_ROOT / datetime.now().strftime("run_%Y%m%d_%H%M%S_notebook")
run_directory.mkdir(parents=True, exist_ok=False)

model_cpu = model.cpu().eval()
checkpoint = {
    "model_state_dict": model_cpu.state_dict(),
    "input_size": training.INPUT_SIZE,
    "hidden_sizes": [128, 64],
    "class_labels": training.CLASS_LABELS,
    "dropout": config.dropout,
    "feature_mean": torch.tensor(feature_mean),
    "feature_std": torch.tensor(feature_std),
    "preprocessing": {
        "center_landmark": 0,
        "scale_landmark": 9,
        "align_palm_to_negative_y": True,
        "clip_range": [-4.0, 4.0],
    },
}
torch.save(checkpoint, run_directory / "left_hand_model.pt")

scripted_model = torch.jit.trace(
    model_cpu,
    torch.zeros(1, training.INPUT_SIZE, dtype=torch.float32),
)
scripted_model.save(str(run_directory / "left_hand_model_torchscript.pt"))

np.savez(
    run_directory / "preprocessing.npz",
    feature_mean=feature_mean,
    feature_std=feature_std,
    class_labels=np.array(training.CLASS_LABELS),
)

metrics = {
    "dataset_size": len(frame),
    "class_counts": {str(k): int(v) for k, v in class_counts.items()},
    "device": str(DEVICE),
    "best_epoch": best_epoch,
    "completed_epochs": len(history["train_loss"]),
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "test_macro_f1": macro_f1,
    "config": asdict(config),
    "warning": "Kişi/session metadatası olmadığı için split temporal bloklarla oluşturuldu.",
}
(run_directory / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)
(run_directory / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
report_frame.to_csv(run_directory / "classification_report.csv")
split_summary.to_csv(run_directory / "split_summary.csv", index=False)
thresholds.to_csv(run_directory / "confidence_thresholds.csv", index=False)

# Hatalı örnekleri UUID ve fotoğraf yoluyla inceleyebilmek için tahminleri kaydet.
test_rows = frame.loc[splits["test"], ["id", "label", "image_path", "captured_at"]].reset_index(drop=True)
test_rows["predicted_label"] = np.array(training.CLASS_LABELS)[test_predictions]
test_rows["confidence"] = test_probabilities.max(axis=1)
for class_index, class_label in enumerate(training.CLASS_LABELS):
    test_rows[f"probability_{class_label}"] = test_probabilities[:, class_index]
test_rows.to_csv(run_directory / "test_predictions.csv", index=False)

training.save_training_history(history, run_directory / "training_history.png")
training.save_confusion_matrix(
    test_targets, test_predictions, run_directory / "confusion_matrix.png"
)
training.save_confidence_plots(
    test_targets, test_probabilities, run_directory / "confidence_analysis.png"
)

print(f"Tüm çıktılar kaydedildi: {run_directory}")

Tüm çıktılar kaydedildi: C:\Users\Ulvi\Desktop\ChordFlow\model\training_outputs\left_hand\run_20260729_191259_notebook


## Sonraki adım

Notebook sonuçlarını değerlendirirken yalnızca genel accuracy'ye bakma:

- Confusion matrix üzerinden karışan dereceleri belirle.
- `test_predictions.csv` ile hatalı fotoğrafları UUID/path üzerinden incele.
- Confidence threshold tablosunda accuracy–coverage dengesini kontrol et.
- Gerçek genelleme için farklı günlerde ve farklı kişilerden veri topla.
- Model yeterli olduğunda ONNX'e dönüştürüp React uygulamasındaki classifier worker'a bağla.

## 7. Tarayıcı için ONNX export

Bu hücre preprocessing adımlarını da model grafiğine gömerek son checkpointi `public/models/left_hand_model.onnx` konumuna aktarır. React sayfası ham 63 MediaPipe koordinatını doğrudan bu modele verir.

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        str(MODEL_DIR / "export_left_hand_onnx.py"),
        "--checkpoint",
        str(run_directory / "left_hand_model.pt"),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, str(MODEL_DIR / "verify_left_hand_onnx.py")],
    cwd=PROJECT_ROOT,
    check=True,
)

print("Realtime test: http://localhost:5173/classify")